In [45]:
import sys
sys.path.append(
    "D:\\Quan\\roboDK\\Vision-Machine-collab-Nhan\\VIKO_UltraRobot"
)
from config import config as CFG

In [2]:
pwd

'd:\\Quan\\roboDK\\Vision-Machine-collab-Nhan\\VIKO_UltraRobot\\src'

In [3]:
from ultralytics import YOLO
model = YOLO(CFG.MODEL['YOLOV9']['WEIGHT'])

In [12]:

import cv2
path_img = '../runs/segment/predict38/image0.jpg'
img = cv2.imread(path_img, cv2.IMREAD_COLOR)
img_cvt = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resize = cv2.resize(img, (640, 640), interpolation= cv2.INTER_CUBIC)

dict_re = model.predict(img_resize, save = True)


0: 640x640 1 weld, 149.5ms
Speed: 5.0ms preprocess, 149.5ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs\segment\predict


In [56]:
from ultralytics import YOLO
import cv2
import pandas as pd
from datetime import datetime
import os

# Khởi tạo mô hình YOLO với trọng số từ CFG
model = YOLO(CFG.MODEL['YOLOV9']['WEIGHT'])

# Đọc ảnh từ đường dẫn
path_img = '../3weld.png'
img = cv2.imread(path_img, cv2.IMREAD_COLOR)
img_cvt = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resize = cv2.resize(img, (640, 640), interpolation=cv2.INTER_CUBIC)


results = model.predict(img_resize, save=True)

def save_data_predict(results):
    csv_file = 'predictions.csv'
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
    else:
        df = pd.DataFrame(columns=['Timestamp', 'ImageName', 'OutputPath', 'Label', 'Confidence'])

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    output_dir = '../runs/segment/predict38/output'
    os.makedirs(output_dir, exist_ok=True)


    new_rows = []
    labels = []


    for idx, result in enumerate(results):
        boxes = result.boxes  
        for box in boxes:
            label = int(box.cls.item()) 
            confidence = float(box.conf.item()) 
            
            image_name = f"sample_{len(df) + len(new_rows) + 1}.png"
            output_path = os.path.join(output_dir, image_name)
            cv2.imwrite(output_path, img_resize)

            new_rows.append({
                'Timestamp': timestamp,
                'ImageName': image_name,
                'OutputPath': output_path,
                'Label': label,
                'Confidence': confidence
            })
            labels.append(label)

    new_df = pd.DataFrame(new_rows)
    df = pd.concat([df, new_df], ignore_index=True)

    df.to_csv(csv_file, index=False)
    return labels

save_data_predict(results= results)




0: 640x640 3 welds, 33.0ms
Speed: 1.0ms preprocess, 33.0ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs\segment\predict11


In [43]:
dict_re.

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: ultralytics.engine.results.Masks object
 names: {0: 'weld', 1: 'other'}
 obb: None
 orig_img: array([[[ 71,  79,  69],
         [ 70,  78,  68],
         [ 68,  77,  67],
         ...,
         [ 81,  86,  77],
         [ 84,  87,  78],
         [ 90,  93,  84]],
 
        [[ 67,  75,  65],
         [ 69,  77,  67],
         [ 69,  78,  68],
         ...,
         [ 81,  86,  77],
         [ 83,  86,  77],
         [ 93,  96,  87]],
 
        [[ 66,  74,  64],
         [ 69,  77,  67],
         [ 70,  79,  69],
         ...,
         [ 82,  87,  78],
         [ 84,  87,  78],
         [ 96,  99,  90]],
 
        ...,
 
        [[116, 109,  92],
         [120, 113,  96],
         [117, 110,  93],
         ...,
         [126, 124, 106],
         [134, 127, 110],
         [140, 133, 116]],
 
        [[117, 111,  92],
         [121, 115,  96],
         [118

In [26]:
box_coordinate = dict_re[0].boxes.xyxy.detach().cpu().numpy().astype('uint32')

In [28]:
x_end, y_end = box_coordinate[0][2], box_coordinate[0][3]

In [29]:
arr = dict_re[0].masks.xy[0]

In [34]:
import numpy as np

def find_point_end(arr):
    points = arr

    start_point = points[0]

    distances = np.linalg.norm(points - start_point, axis=1)

    max_distance_index = np.argmax(distances)

    max_distance_point = points[max_distance_index]

    return max_distance_point, distances[max_distance_index]


In [35]:
coor_end = find_point_end(arr)

In [38]:
coor_end[0][0]

330.0

In [31]:
img_re = dict_re[0][0].orig_img

In [32]:
import plotly.express as px 

px.imshow(img_re)

In [7]:
import numpy as np
def convert_mask(mask):
    color_image = np.zeros((640, 640, 3), dtype=np.uint8)
    color_image[:, :, 0] = mask  
    color_image[:, :, 1] = mask  
    color_image[:, :, 2] = mask  
    return color_image

In [8]:
mask = dict_re[0].masks.data.detach().cpu().numpy()

In [9]:
mask_g = mask[0] + mask[1] + mask[2]

IndexError: index 1 is out of bounds for axis 0 with size 1

In [100]:
mask_g_cvt = cv2.cvtColor(mask_g, cv2.COLOR_GRAY2BGR)

In [80]:
# mask_g = cv2.normalize(mask_g, None, 255, 0, cv2.NORM_MINMAX, cv2.CV_8U)
# mask_g = convert_mask(mask_g)

In [15]:
cv2.imshow('Output', dict_re[0].plot())
cv2.waitKey(0)
cv2.destroyAllWindows()

In [97]:
img_merge = img_re + mask_g_cvt

In [102]:
mask_g_cvt.shape


(640, 640, 3)

In [101]:
cv2.imwrite('test.png', mask_g_cvt)

True

In [90]:
# display 1
import numpy as np
out = np.hstack([img_re, mask_g_cvt])

# Now show the image
cv2.imshow('Output', mask_g_cvt)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [10]:
#display 2
coordinate_re = dict_re[0].masks.xy[0]

img_cp = img_re.copy()
pts = np.zeros((len(coordinate_re), 2))

for idx, point in enumerate(coordinate_re):
    cv2.circle(img_cp, (int(point[0]), int(point[1])), 1, (255, 0, 0), 1)
    pts[idx] = [int(point[0]), int(point[1])]
pts = np.array(pts, np.int32)

id = 0
cv2.fillPoly(img_cp, pts=[pts], color=(0, 0, 100 + int(id) * 100, 20))

array([[[ 71,  79,  69],
        [ 70,  78,  68],
        [ 68,  77,  67],
        ...,
        [ 81,  86,  77],
        [ 84,  87,  78],
        [ 90,  93,  84]],

       [[ 67,  75,  65],
        [ 69,  77,  67],
        [ 69,  78,  68],
        ...,
        [ 81,  86,  77],
        [ 83,  86,  77],
        [ 93,  96,  87]],

       [[ 66,  74,  64],
        [ 69,  77,  67],
        [ 70,  79,  69],
        ...,
        [ 82,  87,  78],
        [ 84,  87,  78],
        [ 96,  99,  90]],

       ...,

       [[116, 109,  92],
        [120, 113,  96],
        [117, 110,  93],
        ...,
        [126, 124, 106],
        [134, 127, 110],
        [140, 133, 116]],

       [[117, 111,  92],
        [121, 115,  96],
        [118, 112,  93],
        ...,
        [134, 132, 114],
        [136, 129, 112],
        [140, 133, 116]],

       [[135, 129, 110],
        [129, 123, 104],
        [130, 124, 105],
        ...,
        [133, 131, 113],
        [142, 135, 118],
        [156, 149, 132]]

In [11]:
cv2.imshow('Image', img_cp)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [30]:
coordinate_re[0]


array([        105,         176], dtype=float32)

In [31]:
coordinate_re[200]

array([        418,         510], dtype=float32)